In [23]:
import re

## Regular Expressions & Tokenization Mechanics

Focus: Pattern Compilation, Groups, Quantifiers & the re Engine

CPython's re module uses a backtracking NFA (Nondeterministic Finite Automaton) engine.

## How to Form Regex Patterns

Building a regex pattern is not about guessing weird punctuation—it is about compiling an exact blueprint of rules that the regex engine steps through character-by-character from left to right.

## Part 1: The 4 Core Building Blocks

Every regex pattern is assembled from 4 categories:

- Literal Characters
- Character Sets / Classes
- Quantifiers
- Anchors / Groups

### 1. Literals (Exact Matches)

Any alphanumeric character matches itself literally.

- Pattern: `ALLOC:` → matches the exact characters A, L, L, O, C, : in that order.

**Special Characters (Metacharacters):**

These 12 characters have special engine meanings: `^ $ . * + ? ( ) [ ] { } | \`

Rule: If you want to match a literal dot or parenthesis, escape it with a backslash `\`:

- Literal period: `\.`
- Literal parenthesis: `\(` and `\)`
- Literal bracket: `\[` and `\]`

### 2. Character Classes (What type of character is allowed?)

Instead of matching one exact letter, you define a bucket of allowed characters:

- `\d`: Any single digit 0-9
- `\w`: Any word character (a-z, A-Z, 0-9, _)
- `\s`: Any whitespace character (space, tab `\t`, newline `\n`)
- `.`: Any character whatsoever (except newline)
- `[0-9a-fA-F]`: A custom set (here: any valid hex digit)
- `[^0-9]`: Negation set (anything that is not a digit)

### 3. Quantifiers (How many times does it repeat?)

A quantifier always attaches to the single character/group immediately to its left:

- `+`: 1 or more times (e.g., `\d+` = one or more digits like 4 or 4096)
- `*`: 0 or more times (optional repetition)
- `?`: 0 or 1 time (strictly optional)
- `{4}`: Exactly 4 times (e.g., `[0-9a-f]{4}` matches 7ffe)
- `{2,8}`: Between 2 and 8 times

### 4. Groups & Captures (Structure & Memory)

- `(pattern)`: Positional Capture Group (saves what's inside)
- `(?P<name>pattern)`: Named Capture Group (saves what's inside with a key)
- `(?:pattern)`: Non-Capturing Group (applies logic/quantifiers, but saves no memory)

## Part 2: The Raw String Rule (`r"..."`)

In Python, standard strings use `\` for escape sequences (e.g., `\n` is newline, `\t` is tab, `\b` is backspace).

If you write:
```python
pattern = "\b\d+\b"  # Python interprets \b as ASCII backspace!
```

The regex engine never sees the word-boundary command `\b`.

**The Iron Rule:** Always prefix regex patterns with `r` (raw string literal) in Python.

```python
pattern = r"\b\d+\b"  # Tells Python: do not interpret backslashes, pass them raw to the regex engine.
```

## Part 3: Step-by-Step Blueprint Construction

Let's construct the pattern for a real-world log line step-by-step.

**Target String:**
```
ALLOC: buffer_id=0x7ffe ptr=0x1000 size=4096 bytes (flags: READ|WRITE)
```

**Step 1: Break the target into static vs. dynamic tokens**

| Section in String | Type | What should the rule be? |
|---|---|---|
| `ALLOC: buffer_id=` | Exact static prefix | Literal text: `ALLOC:\s+buffer_id=` |
| `0x7ffe` | Dynamic Hex Value | Capture group for hex: `(?P<buffer_id>0x[0-9a-fA-F]+)` |
| ` ptr=` | Static spacer | Literal space and text: `\s+ptr=` |
| `0x1000` | Dynamic Hex Value | Capture group for hex: `(?P<ptr>0x[0-9a-fA-F]+)` |
| ` size=` | Static spacer | Literal space and text: `\s+size=` |
| `4096` | Dynamic Integer | Capture group for digits: `(?P<size>\d+)` |
| ` bytes` | Static spacer | Literal text: `\s+bytes` |
| `(flags: READ\|WRITE)` | Optional metadata | Non-capturing group: `(?:\s+\(flags:\s+[A-Z\|]+\))?` |

**Step 2: Assemble the Pieces into the Final Pattern**

```python
pattern = (
    r"ALLOC:\s+buffer_id="                 # Static prefix
    r"(?P<buffer_id>0x[0-9a-fA-F]+)"       # Capture group 1: hex address
    r"\s+ptr="                             # Static spacer
    r"(?P<ptr>0x[0-9a-fA-F]+)"             # Capture group 2: hex address
    r"\s+size="                            # Static spacer
    r"(?P<size>\d+)"                       # Capture group 3: integer size
    r"\s+bytes"                            # Static spacer
    r"(?:\s+\(flags:\s+[A-Z|]+\))?"        # Non-capturing optional group
)
```

## Part 4: The Execution Code

Now, run this against the string in Python and watch the output:

```python
import re

log_line = "ALLOC: buffer_id=0x7ffe ptr=0x1000 size=4096 bytes (flags: READ|WRITE)"

pattern = re.compile(
    r"ALLOC:\s+buffer_id=(?P<buffer_id>0x[0-9a-fA-F]+)\s+"
    r"ptr=(?P<ptr>0x[0-9a-fA-F]+)\s+"
    r"size=(?P<size>\d+)\s+bytes"
    r"(?:\s+\(flags:\s+[A-Z|]+\))?"
)

match = pattern.search(log_line)

if match:
    data = match.groupdict()
    # Cast size to integer manually
    data["size"] = int(data["size"])
    
    print("Parsed Dictionary:")
    print(data)
```

**Output:**
```
{'buffer_id': '0x7ffe', 'ptr': '0x1000', 'size': 4096}
```

**Notice what happened:**

- `buffer_id`, `ptr`, and `size` were extracted into the dictionary with explicit keys.
- The flags portion `(flags: READ|WRITE)` was matched by the engine to validate the end of the line.
- Because of `(?:...)`, it did not waste memory creating a key or slot in `groupdict()`.

In [24]:
pattern = (
    r"ALLOC:\s+buffer_id="                 # Static prefix
    r"(?P<buffer_id>0x[0-9a-fA-F]+)"       # Capture group 1: hex address
    r"\s+ptr="                             # Static spacer
    r"(?P<ptr>0x[0-9a-fA-F]+)"              # Capture group 2: hex address
    r"\s+size="                            # Static spacer
    r"(?P<size>\d+)"                       # Capture group 3: integer size
    r"\s+bytes"                            # Static spacer
    r"(?:\s+\(flags:\s+[A-Z|]+\))?"        # Non-capturing optional group
)

log_line = "ALLOC: buffer_id=0x7ffe ptr=0x1000 size=4096 bytes (flags: READ|WRITE)"

pattern = re.compile(
    r"ALLOC:\s+buffer_id=(?P<buffer_id>0x[0-9a-fA-F]+)\s+"
    r"ptr=(?P<ptr>0x[0-9a-fA-F]+)\s+"
    r"size=(?P<size>\d+)\s+bytes"
    r"(?:\s+\(flags:\s+[A-Z|]+\))?"
)

match = pattern.search(log_line)

if match:
    data = match.groupdict()
    # Cast size to integer manually
    data["size"] = int(data["size"])
    
    print("Parsed Dictionary:")
    print(data)

Parsed Dictionary:
{'buffer_id': '0x7ffe', 'ptr': '0x1000', 'size': 4096}


In [25]:
import re

error_log = "CUDA_ERROR: [device:0] launch failed on kernel 'vec_add' with error 702 (status: ILLEGAL_MEMORY_ACCESS)"

pattern = re.compile(
    r"CUDA_ERROR:\s+\[device:(?P<device_id>\d+)\]\s+"    # device_id (digits)
    r"launch failed on kernel\s+'(?P<kernel_name>\w+)'\s+" # kernel_name
    r"with error\s+(?P<error_code>\d+)\s+"               # error_code (digits)
    r"(?:\(status:\s+[A-Z_]+\))"                         # non-capturing status
)

match = pattern.search(error_log)

if match:
    data = match.groupdict()
    # Cast numeric values
    data["device_id"] = int(data["device_id"])
    data["error_code"] = int(data["error_code"])
    print("Successfully parsed:")
    print(data)

Successfully parsed:
{'device_id': 0, 'kernel_name': 'vec_add', 'error_code': 702}


## Iterating the String

In [26]:
import re

text = "Log 01: [#42] succeeded. Log 02: [#108] failed."

## Search vs. Match vs. Findall vs. Finditer

- `re.match()`: Anchored at the start of the string only (equivalent to `^pattern`).
- `re.search()`: Scans the entire string to find the first location where the pattern matches.
- `re.findall()`: Returns a list of all matching strings or tuples of groups. (Eager: loads all into memory at once).
- `re.finditer()`: Returns an iterator yielding Match objects lazily. Memory efficient O(1) and gives match metadata (`start()`, `end()`, `span()`, `groups()`).

1. re.match() — "Does it start here?"

match() only looks at the beginning of the string.

In [27]:
result = re.match(r"Log", text)

print(result)

<re.Match object; span=(0, 3), match='Log'>


2. re.search() — "Find the first one anywhere"

Unlike match(), search() scans through the string.

In [28]:
result = re.search(r"\[#\d+\]", text)

print(result.group())
print(result.start())
print(result.end())
print(result.span())

[#42]
8
13
(8, 13)


3. re.findall() — "Give me everything"

Now suppose we want all the tokens.

In [29]:
result = re.findall(r"\[#\d+\]", text)

print(result)

['[#42]', '[#108]']


Because we put `\d+` inside `(...)`, `findall()` returns the captured group, not the entire `[#42]`.

- `r"\[#\d+\]"` gives: `['[#42]', '[#108]']`
- `r"\[#(\d+)\]"` gives: `['42', '108']`

4. re.finditer() — "Give me every Match object"

This is similar to findall(), but instead of giving you strings, it gives you Match objects.

In [30]:
matches = re.finditer(r"\[#(\d+)\]", text)

for match in matches:
    print(match)

<re.Match object; span=(8, 13), match='[#42]'>
<re.Match object; span=(33, 39), match='[#108]'>


In [31]:
for match in re.finditer(r"\[#(\d+)\]", text):
    print("Full match:", match.group())
    print("Number:", match.group(1))
    print("Start:", match.start())
    print("End:", match.end())
    print("Span:", match.span())
    print()

Full match: [#42]
Number: 42
Start: 8
End: 13
Span: (8, 13)

Full match: [#108]
Number: 108
Start: 33
End: 39
Span: (33, 39)



finditer() is lazy: it doesn't create a list containing every match up front.

## The Real Problem: Matching vs. Extracting

## 1. Positional Groups `()`: The Tape Recorder

When you wrap parentheses `()` around any part of a regex, you tell the engine:

> "Match the whole pattern, but remember whatever matched inside this specific set of parentheses and drop it into a numbered bucket."

### The Real-World Example: Parsing a CUDA Launch Configuration

Suppose you're parsing a kernel launch configuration string:
```
config = "<<<grid=(128, 1), block=(64, 4)>>>"
```

You want the dimensions out as raw numbers.

```python
import re

text = "<<<grid=(128, 1), block=(64, 4)>>>"

# We put parentheses around the parts we want to save
pattern = r"<<<grid=\((\d+),\s*(\d+)\),\s*block=\((\d+),\s*(\d+)\)>>>"

match = re.search(pattern, text)

if match:
    # group(0) is always the ENTIRE matched string
    print(f"Full match: {match.group(0)}")
    
    # group(1..4) correspond to the open parentheses from left to right
    grid_x  = int(match.group(1))  # 128
    grid_y  = int(match.group(2))  # 1
    block_x = int(match.group(3))  # 64
    block_y = int(match.group(4))  # 4
    
    total_threads = grid_x * grid_y * block_x * block_y
    print(f"Total spawned threads: {total_threads}")
```

### The Pain Point with Positional Groups

What happens when your colleague edits the regex and inserts an optional prefix group at the front `r"((cuda|triton)\s+)?<<<grid..."`?

Now `match.group(1)` is no longer `grid_x`—it's `"cuda"`. Every single index in your codebase is shifted by 1 or 2, and your pipeline silently produces garbage data.

This brings us to...

## 2. Named Groups `(?P<name>...)`): Dictionaries Direct from the Engine

Instead of counting parentheses on your fingers, you give each bucket an explicit variable name directly inside the pattern using the syntax:
```
(?P<your_variable_name>sub_pattern)
```

### Rewriting the Same Example with Named Groups

```python
import re

text = "<<<grid=(128, 1), block=(64, 4)>>>"

pattern = re.compile(
    r"<<<grid=\((?P<gx>\d+),\s*(?P<gy>\d+)\),\s*block=\((?P<bx>\d+),\s*(?P<by>\d+)\)>>>"
)

match = pattern.search(text)
if match:
    # 1. Fetch by explicit name - zero guessing
    print(match.group("gx"))  # "128"
    print(match.group("bx"))  # "64"

    # 2. Or dump the entire match straight into a Python dictionary!
    data = match.groupdict()
    print(data)
    # Output: {'gx': '128', 'gy': '1', 'bx': '64', 'by': '4'}
```

Now you don't care if somebody reorders or adds groups elsewhere in the pattern—`match.group("gx")` will always target the grid dimension.

## 3. Non-Capturing Groups `(?:...)`: The Ghost Group

Sometimes you need parentheses for logic (like grouping an OR condition or applying a quantifier `+`), but you do not want to save the result.

Every time you use a standard `()`, the CPython regex engine allocates a memory slot on the match object to save that slice of the string. In hot loops processing millions of log tokens, allocating unneeded capture groups kills throughput.

### The Problem

Suppose you want to match URLs that start with either `http://` or `https://`:

```python
# BAD: standard capturing group
pattern_bad = r"(http|https)://([\w\.]+)"
m = re.search(pattern_bad, "https://github.com")
print(m.groups())  # ('https', 'github.com') -> 'https' wasted a capture slot!
```

### The Fix

Add `?:` right after the opening parenthesis:

`(?:\w+)` tells the engine: "Use this for grouping logic, but DO NOT allocate memory to capture it."

```python
# GOOD: Non-capturing group for the protocol, capturing group for the domain
pattern_good = r"(?:http|https)://([\w\.]+)"
m = re.search(pattern_good, "https://github.com")
print(m.groups())  # ('github.com',) -> Clean! Only the data you actually need.
```

## 4. Greedy vs. Non-Greedy: Pac-Man vs. The Scalpel

This is the single most common source of regex bugs in production.

By default, repetition quantifiers (`*`, `+`, `{m,n}`) are **Greedy**. They will eat as many characters as physically possible across the string until they hit the end, and only backtrack character-by-character when the rest of the pattern fails.

### Let's Watch the Engine's Brain

Take this string containing two tensor definitions:
```
text = "Tensor(shape=[32, 64]) and Tensor(shape=[128, 256])"
```

You want to extract individual tensors: `"Tensor(...)"`.

### The Greedy Mistake (`.*`)

```python
pattern = r"Tensor\(.*\)"
match = re.findall(pattern, text)
print(match)
```

**Output:**
```
['Tensor(shape=[32, 64]) and Tensor(shape=[128, 256])']
```

**Why did it return one giant string instead of two?**

- The engine hits `Tensor(`.
- `.*` is Greedy—it says: "I will eat everything all the way to the very end of the string."
- At the end of the string, it looks for `)`. The last character is `)`, so it claims victory and stops. It swallowed the middle of your data whole!

```
[Tensor(shape=[32, 64]) and Tensor(shape=[128, 256])]
 ▲                                                 ▲
 Starts here ------------------------ Eats to here!
```

### The Lazy / Non-Greedy Solution (`.*?`)

Adding a `?` after any quantifier (`*?`, `+?`, `??`) turns it into **Lazy** mode.

It tells the engine: "Consume the absolute bare minimum number of characters required to satisfy the match, then stop immediately."

```python
pattern = r"Tensor\(.*?\)"
match = re.findall(pattern, text)
print(match)
```

**Output:**
```
['Tensor(shape=[32, 64])', 'Tensor(shape=[128, 256])']
```

```
[Tensor(shape=[32, 64])] and [Tensor(shape=[128, 256])]
 ▲                    ▲      ▲                      ▲
 Match 1 stops here --┘      Match 2 stops here ----┘
```